<a href="https://colab.research.google.com/github/udlbook/udlbook/blob/main/Notebooks/Chap10/10_3_2D_Convolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 10.3: 2D Convolution**

This notebook investigates the 2D convolution operation.  It asks you to hand code the convolution so we can be sure that we are computing the same thing as in PyTorch.  The next notebook uses the convolutional layers in PyTorch directly.

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

Contact me at udlbookmail@gmail.com if you find any mistakes or have any suggestions.

In [1]:
import numpy as np
import torch
# Set to print in reasonable form
np.set_printoptions(precision=3, floatmode="fixed")
torch.set_printoptions(precision=3)

This routine performs convolution in PyTorch

In [2]:
# Perform convolution in PyTorch
def conv_pytorch(image, conv_weights, stride=1, pad =1):
  # Convert image and kernel to tensors
  image_tensor = torch.from_numpy(image) # (batchSize, channelsIn, imageHeightIn, =imageWidthIn)
  conv_weights_tensor = torch.from_numpy(conv_weights) # (channelsOut, channelsIn, kernelHeight, kernelWidth)
  # Do the convolution
  output_tensor = torch.nn.functional.conv2d(image_tensor, conv_weights_tensor, stride=stride, padding=pad)
  # Convert back from PyTorch and return
  return(output_tensor.numpy()) # (batchSize channelsOut imageHeightOut imageHeightIn)

First we'll start with the simplest 2D convolution.  Just one channel in and one channel out.  A single image in the batch.

In [3]:
# Perform convolution in numpy
def conv_numpy_1(image, weights, pad=1):

    # Perform zero padding
    if pad != 0:
        image = np.pad(image, ((0, 0), (0 ,0), (pad, pad), (pad, pad)),'constant')

    # Get sizes of image array and kernel weights
    batchSize,  channelsIn, imageHeightIn, imageWidthIn = image.shape
    channelsOut, channelsIn, kernelHeight, kernelWidth = weights.shape

    # Get size of output arrays
    imageHeightOut = np.floor(1 + imageHeightIn - kernelHeight).astype(int)
    imageWidthOut = np.floor(1 + imageWidthIn - kernelWidth).astype(int)

    # Create output
    out = np.zeros((batchSize, channelsOut, imageHeightOut, imageWidthOut), dtype=np.float32)

    # !!!!!! NOTE THERE IS A SUBTLETY HERE !!!!!!!!
    # I have padded the image with zeros above, so it is surrouned by a "ring" of zeros
    # That means that the image indexes are all off by one
    # This actually makes your code simpler

    for c_y in range(imageHeightOut):
      for c_x in range(imageWidthOut):
        for c_kernel_y in range(kernelHeight):
          for c_kernel_x in range(kernelWidth):
            # TODO -- Retrieve the image pixel and the weight from the convolution
            # Only one image in batch, one input channel and one output channel, so these indices should all be zero
            # Replace the two lines below
            this_pixel_value = image[0,0,c_y+c_kernel_y,c_x+c_kernel_x]
            this_weight = weights[0,0,c_kernel_y,c_kernel_x]


            # Multiply these together and add to the output at this position
            out[0, 0, c_y, c_x] += np.sum(this_pixel_value * this_weight)

    return out

In [4]:
# Set random seed so we always get same answer
np.random.seed(1)
n_batch = 1
image_height = 4
image_width = 6
channels_in = 1
kernel_size = 3
channels_out = 1

# Create random input image
input_image= np.random.normal(size=(n_batch, channels_in, image_height, image_width))
# Create random convolution kernel weights
conv_weights = np.random.normal(size=(channels_out, channels_in, kernel_size, kernel_size))

# Perform convolution using PyTorch
conv_results_pytorch = conv_pytorch(input_image, conv_weights, stride=1, pad=1)
print("PyTorch Results")
print(conv_results_pytorch)

# Perform convolution in numpy
print("Your results")
conv_results_numpy = conv_numpy_1(input_image, conv_weights)
print(conv_results_numpy)

PyTorch Results
[[[[-0.92877074 -2.76029052  0.71617666  0.11420725  0.5596409
    -0.38718181]
   [-1.51472398  0.28314694  1.00809468  0.46584486 -1.0939884
     2.00448021]
   [-1.63398828  3.55485207 -2.15395759 -0.89198292 -1.85607063
     2.29928377]
   [ 0.56543283 -0.94654296 -0.62942895  2.99601051 -1.81129165
    -0.53341   ]]]]
Your results
[[[[-0.9287708  -2.7602906   0.7161767   0.1142073   0.5596409
    -0.38718182]
   [-1.5147239   0.28314698  1.0080947   0.46584484 -1.0939885
     2.0044804 ]
   [-1.6339881   3.554852   -2.1539576  -0.8919829  -1.8560706
     2.2992837 ]
   [ 0.56543285 -0.946543   -0.629429    2.9960103  -1.8112916
    -0.53341   ]]]]


Let's now add in the possibility of using different strides

In [5]:
# Perform convolution in numpy
def conv_numpy_2(image, weights, stride=1, pad=1):

    # Perform zero padding
    if pad != 0:
        image = np.pad(image, ((0, 0), (0 ,0), (pad, pad), (pad, pad)),'constant')

    # Get sizes of image array and kernel weights
    batchSize,  channelsIn, imageHeightIn, imageWidthIn = image.shape
    channelsOut, channelsIn, kernelHeight, kernelWidth = weights.shape

    # Get size of output arrays
    imageHeightOut = np.floor(1 + (imageHeightIn - kernelHeight) / stride).astype(int)
    imageWidthOut = np.floor(1 + (imageWidthIn - kernelWidth) / stride).astype(int)

    # Create output
    out = np.zeros((batchSize, channelsOut, imageHeightOut, imageWidthOut), dtype=np.float32)

    for c_y in range(imageHeightOut):
      for c_x in range(imageWidthOut):
        for c_kernel_y in range(kernelHeight):
          for c_kernel_x in range(kernelWidth):
            # TODO -- Retrieve the image pixel and the weight from the convolution
            # Only one image in batch, one input channel and one output channel, so these indices should all be zero
            # Replace the two lines below
            this_pixel_value = image[0,0,c_y*stride+c_kernel_y,c_x*stride+c_kernel_x]
            this_weight = weights[0,0,c_kernel_y,c_kernel_x]


            # Multiply these together and add to the output at this position
            out[0, 0, c_y, c_x] += np.sum(this_pixel_value * this_weight)

    return out

In [8]:
# Set random seed so we always get same answer
np.random.seed(1)
n_batch = 1
image_height = 12
image_width = 10
channels_in = 1
kernel_size = 3
channels_out = 1
stride = 2

# Create random input image
input_image= np.random.normal(size=(n_batch, channels_in, image_height, image_width))
# Create random convolution kernel weights
conv_weights = np.random.normal(size=(channels_out, channels_in, kernel_size, kernel_size))

# Perform convolution using PyTorch
conv_results_pytorch = conv_pytorch(input_image, conv_weights, stride, pad=1)
print("PyTorch Results")
print(conv_results_pytorch)

# Perform convolution in numpy
print("Your results")
conv_results_numpy = conv_numpy_2(input_image, conv_weights, stride, pad=1)
print(conv_results_numpy)

PyTorch Results
[[[[-0.80936502 -4.55000504 -5.48644408 -9.50590007 -4.5119328 ]
   [-0.05546526  1.14484207 -5.38831003 -3.9102454   0.09650551]
   [-0.18614325  0.65958396  1.62966335  2.27526627  4.87444365]
   [ 2.38590982 -0.22541781  3.28823207 -4.23915446 -1.4026945 ]
   [ 0.82496306  1.71026929 -3.24597272  3.24605864  1.7087309 ]
   [ 0.8088478   3.69534498  3.49057599 -2.11278279 -2.71367209]]]]
Your results
[[[[-0.8093651  -4.550005   -5.4864435  -9.505899   -4.511933  ]
   [-0.05546521  1.1448419  -5.3883104  -3.9102452   0.09650555]
   [-0.18614332  0.65958387  1.6296635   2.2752664   4.8744435 ]
   [ 2.38591    -0.22541776  3.288232   -4.2391543  -1.4026946 ]
   [ 0.82496303  1.7102693  -3.2459726   3.2460587   1.7087309 ]
   [ 0.8088478   3.6953452   3.4905758  -2.112783   -2.713672  ]]]]


Now we'll introduce multiple input and output channels

In [9]:
# Perform convolution in numpy
def conv_numpy_3(image, weights, stride=1, pad=1):

    # Perform zero padding
    if pad != 0:
        image = np.pad(image, ((0, 0), (0 ,0), (pad, pad), (pad, pad)),'constant')

    # Get sizes of image array and kernel weights
    batchSize,  channelsIn, imageHeightIn, imageWidthIn = image.shape
    channelsOut, channelsIn, kernelHeight, kernelWidth = weights.shape

    # Get size of output arrays
    imageHeightOut = np.floor(1 + (imageHeightIn - kernelHeight) / stride).astype(int)
    imageWidthOut = np.floor(1 + (imageWidthIn - kernelWidth) / stride).astype(int)

    # Create output
    out = np.zeros((batchSize, channelsOut, imageHeightOut, imageWidthOut), dtype=np.float32)

    for c_y in range(imageHeightOut):
      for c_x in range(imageWidthOut):
        for c_channel_out in range(channelsOut):
          for c_channel_in in range(channelsIn):
            for c_kernel_y in range(kernelHeight):
              for c_kernel_x in range(kernelWidth):
                  # TODO -- Retrieve the image pixel and the weight from the convolution
                  # Only one image in batch so this index should be zero
                  # Replace the two lines below
                  this_pixel_value = image[0,c_channel_in,c_y*stride+c_kernel_y,c_x*stride+c_kernel_x]
                  this_weight = weights[c_channel_out,c_channel_in,c_kernel_y,c_kernel_x]

                  # Multiply these together and add to the output at this position
                  out[0, c_channel_out, c_y, c_x] += np.sum(this_pixel_value * this_weight)
    return out

In [10]:
# Set random seed so we always get same answer
np.random.seed(1)
n_batch = 1
image_height = 4
image_width = 6
channels_in = 5
kernel_size = 3
channels_out = 2

# Create random input image
input_image= np.random.normal(size=(n_batch, channels_in, image_height, image_width))
# Create random convolution kernel weights
conv_weights = np.random.normal(size=(channels_out, channels_in, kernel_size, kernel_size))

# Perform convolution using PyTorch
conv_results_pytorch = conv_pytorch(input_image, conv_weights, stride=1, pad=1)
print("PyTorch Results")
print(conv_results_pytorch)

# Perform convolution in numpy
print("Your results")
conv_results_numpy = conv_numpy_3(input_image, conv_weights, stride=1, pad=1)
print(conv_results_numpy)

PyTorch Results
[[[[ -0.78493005   5.46317843  -2.48025844   5.02562432  -3.59404407
      7.78488108]
   [ -6.74353604   2.53411115  -0.66394352   7.14885746  -9.83852552
      7.8488439 ]
   [ -4.79449808  14.0742739   -1.06040823   2.70604367 -10.18183894
      2.0036554 ]
   [  1.80853737   0.28671588   4.64779395  -1.83962197   3.25877536
      1.0733081 ]]

  [[  4.14990184   5.37204436   1.69949782   0.49957395   0.5894419
      4.36072584]
   [ -4.12345821   5.1360658    4.67700784  -3.89519639  -4.99028029
      2.54604406]
   [  3.99092569   5.76840128  -2.31524793   8.47292739   1.7520073
      2.76562704]
   [  1.52850472   0.3179325   11.51848119  -5.44439849  -2.29293586
      1.26966775]]]]
Your results
[[[[ -0.78493017   5.4631777   -2.4802582    5.025625    -3.594044
      7.784882  ]
   [ -6.7435346    2.5341108   -0.6639434    7.1488576   -9.838525
      7.8488436 ]
   [ -4.794498    14.074275    -1.0604087    2.7060435  -10.181838
      2.003656  ]
   [  1.8085374  

Now we'll do the full convolution with multiple images (batch size > 1), and multiple input channels, multiple output channels.

In [11]:
# Perform convolution in numpy
def conv_numpy_4(image, weights, stride=1, pad=1):

    # Perform zero padding
    if pad != 0:
        image = np.pad(image, ((0, 0), (0 ,0), (pad, pad), (pad, pad)),'constant')

    # Get sizes of image array and kernel weights
    batchSize,  channelsIn, imageHeightIn, imageWidthIn = image.shape
    channelsOut, channelsIn, kernelHeight, kernelWidth = weights.shape

    # Get size of output arrays
    imageHeightOut = np.floor(1 + (imageHeightIn - kernelHeight) / stride).astype(int)
    imageWidthOut = np.floor(1 + (imageWidthIn - kernelWidth) / stride).astype(int)

    # Create output
    out = np.zeros((batchSize, channelsOut, imageHeightOut, imageWidthOut), dtype=np.float32)

    for c_batch in range(batchSize):
      for c_y in range(imageHeightOut):
        for c_x in range(imageWidthOut):
          for c_channel_out in range(channelsOut):
            for c_channel_in in range(channelsIn):
              for c_kernel_y in range(kernelHeight):
                for c_kernel_x in range(kernelWidth):
                    # TODO -- Retrieve the image pixel and the weight from the convolution
                    # Replace the two lines below
                    this_pixel_value = image[c_batch,c_channel_in,c_y*stride+c_kernel_y,c_x*stride+c_kernel_x]
                    this_weight = weights[c_channel_out,c_channel_in,c_kernel_y,c_kernel_x]



                    # Multiply these together and add to the output at this position
                    out[c_batch, c_channel_out, c_y, c_x] += np.sum(this_pixel_value * this_weight)
    return out

In [12]:
# Set random seed so we always get same answer
np.random.seed(1)
n_batch = 2
image_height = 4
image_width = 6
channels_in = 5
kernel_size = 3
channels_out = 2

# Create random input image
input_image= np.random.normal(size=(n_batch, channels_in, image_height, image_width))
# Create random convolution kernel weights
conv_weights = np.random.normal(size=(channels_out, channels_in, kernel_size, kernel_size))

# Perform convolution using PyTorch
conv_results_pytorch = conv_pytorch(input_image, conv_weights, stride=1, pad=1)
print("PyTorch Results")
print(conv_results_pytorch)

# Perform convolution in numpy
print("Your results")
conv_results_numpy = conv_numpy_4(input_image, conv_weights, stride=1, pad=1)
print(conv_results_numpy)

PyTorch Results
[[[[ -3.63266998  -1.64414063   0.16871541  -1.16683534  -3.86470503
      6.04548497]
   [ -9.00441048   7.30303198   4.41414413   0.36094645  -6.73911902
      3.93877819]
   [ -1.39066168  13.50223385   3.80719001  -9.37925647   3.99065754
      5.44193873]
   [  2.80529354   6.87386967  -9.28708597  -4.4677558   -1.50120554
      4.60699744]]

  [[  1.93991531  -1.40994827   2.39733412  -0.23498018  -0.39417446
     -1.48273797]
   [  5.04926785  -3.33537397  -7.59638391  -1.58618293   3.04942231
     -1.85714215]
   [  3.51356752   0.47452556  -1.95244809  -1.29143814  -0.58875724
     -0.94794943]
   [  6.52359837  -0.01988546  -3.29757656  -1.24783696   3.24882183
     -2.67951281]]]


 [[[  4.15358547  -4.76441146  11.63518341   0.50610631  -4.01175338
     -2.08113863]
   [ -1.12514613  -0.67652481  16.74850235  -7.03000422  -5.97797666
     -2.6288367 ]
   [  0.77808623  -3.98359269 -10.28402713   1.57539725  -8.88848701
      1.16275197]
   [  0.55557572  -2.